# Parameter Audit — Copper Forecasting V20 + V21

**Mục tiêu:** dựng lại các kiến trúc đúng theo hai notebook gốc và tính `model.count_params()`, trainable, non-trainable và bảng thông số từng tầng. **Không huấn luyện**, không cần dataset và không chạy Simulated Annealing. Chạy trên Kaggle: **Run All** (TensorFlow/Keras sẵn có).

**Nguồn kiến trúc:** `17-11-Copper-NonSA-Fair-Full-Protocol-V20(1).ipynb` cell `build_net`; `Copper_SA_LSTM_H1_Full_Comparison_Kaggle.ipynb` cell `build_baseline_lstm`, `build_baseline_rnn`, `build_paper_sa_lstm`, `build_sa_lstm`. V21 `Dense(32)` là **giả định reproduction**, không được khẳng định là Dense chính thức của Chen et al. (2023).

**Quy ước:** `LOOKBACK=22`, `HORIZON=1`; Keras 3 / TensorFlow default GRU `reset_after=True`. Optimizer states **không** tính vào model weights. Các chiến lược/ratios/seeds không thay đổi số tham số. V20 Direct và Residual dùng cùng kiến trúc; chỉ target và cách hoàn nguyên dự báo khác nhau.

In [ ]:
import os, json, csv
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

LOOKBACK = 22
HORIZON = 1
OUT = Path('/kaggle/working/copper_v20_v21_parameter_audit' if Path('/kaggle/working').exists() else 'copper_v20_v21_parameter_audit')
OUT.mkdir(parents=True, exist_ok=True)

FEATURE_SETS_V20 = {
    'copper_only': ['Copper'],
    'copper_paper3': ['Copper','WTI','Gold','Silver'],
    'copper_legacy14': ['Copper','Aluminium','Tin','Silver','Gold']
}
FEATURE_SETS_V21 = {
    'paper3_with_copper': ['Copper','WTI','Gold','Silver'],
    'paper3_without_copper': ['WTI','Gold','Silver'],
    'legacy14_with_copper': ['Copper','Aluminium','Tin','Silver','Gold'],
    'legacy14_without_copper': ['Aluminium','Tin','Silver','Gold']
}
PAPER_HP = {'units_1': 39, 'units_2': 111, 'learning_rate': 5e-5}
REOPT_HP = {
    'paper3': {'units_1': 22, 'units_2': 91, 'learning_rate': 0.000922},
    'legacy14': {'units_1': 22, 'units_2': 83, 'learning_rate': 0.000655}
}
print('TensorFlow:', tf.__version__, '| Keras:', tf.keras.__version__ if hasattr(tf.keras,'__version__') else 'bundled')
print('V20 feature sets:', {k:len(v) for k,v in FEATURE_SETS_V20.items()})
print('V21 feature sets:', {k:len(v) for k,v in FEATURE_SETS_V21.items()})

## 1. Kiến trúc V20 (sao chép logic từ `build_net`)

- `lstm` / `gru`: 64 → 32 units → Dense(16) → Dense(1)
- `cnn_lstm` / `cnn_gru`: Causal Conv1D(32,kernel=3) → MaxPool(2) → 48 → 24 units → Dense(16) → Dense(1)
- Dropout(0.2) không có tham số. V20 `ridge_residual` dùng Ridge(alpha=1) trên vector flatten 22×features.

In [ ]:
def build_v20_net(arch, shape):
    inp = tf.keras.Input(shape=shape)
    z = inp
    if arch in ('cnn_lstm','cnn_gru'):
        z = tf.keras.layers.Conv1D(32, 3, padding='causal', activation='relu')(z)
        z = tf.keras.layers.MaxPooling1D(2)(z)
    if arch in ('lstm','cnn_lstm'):
        n1, n2 = (64,32) if arch=='lstm' else (48,24)
        z = tf.keras.layers.LSTM(n1, return_sequences=True)(z)
        z = tf.keras.layers.Dropout(.2)(z)
        z = tf.keras.layers.LSTM(n2)(z)
    elif arch in ('gru','cnn_gru'):
        n1, n2 = (64,32) if arch=='gru' else (48,24)
        z = tf.keras.layers.GRU(n1, return_sequences=True)(z)
        z = tf.keras.layers.Dropout(.2)(z)
        z = tf.keras.layers.GRU(n2)(z)
    else:
        raise ValueError(arch)
    z = tf.keras.layers.Dropout(.2)(z)
    z = tf.keras.layers.Dense(16, activation='relu')(z)
    out = tf.keras.layers.Dense(1, activation='linear')(z)
    return tf.keras.Model(inp, out, name='v20_'+arch)


## 2. Kiến trúc V21 (sao chép logic từ 4 hàm build)

`baseline_lstm`: LSTM(64)→LSTM(32)→Dense(16)→Dense(1); `baseline_rnn`: SimpleRNN(64)→SimpleRNN(32)→Dense(16)→Dense(1).

`paper_sa_lstm`: LSTM(39)→LSTM(111)→Dense(32)→Dense(1). `reopt_sa_lstm`: Paper3 LSTM(22)→LSTM(91), Legacy14 LSTM(22)→LSTM(83), cùng Dense(32)→Dense(1). SA và learning rate là cách chọn siêu tham số, **không tạo thêm trainable weights**.

In [ ]:
def build_v21_net(kind, input_shape, hp=None):
    if kind == 'baseline_lstm':
        m = tf.keras.Sequential(name='Baseline_LSTM')
        m.add(tf.keras.layers.Input(shape=input_shape))
        m.add(tf.keras.layers.LSTM(64, return_sequences=True))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.LSTM(32))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.Dense(16, activation='relu'))
        m.add(tf.keras.layers.Dense(1, activation='linear'))
    elif kind == 'baseline_rnn':
        m = tf.keras.Sequential(name='Baseline_RNN')
        m.add(tf.keras.layers.Input(shape=input_shape))
        m.add(tf.keras.layers.SimpleRNN(64, return_sequences=True))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.SimpleRNN(32))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.Dense(16, activation='relu'))
        m.add(tf.keras.layers.Dense(1, activation='linear'))
    elif kind in ('paper_sa_lstm','reopt_sa_lstm'):
        h = PAPER_HP if kind == 'paper_sa_lstm' else hp
        assert h is not None
        m = tf.keras.Sequential(name='Paper_SA_LSTM' if kind=='paper_sa_lstm' else 'Reoptimized_SA_LSTM')
        m.add(tf.keras.layers.Input(shape=input_shape))
        m.add(tf.keras.layers.LSTM(int(h['units_1']), return_sequences=True, activation='tanh', recurrent_activation='sigmoid'))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.LSTM(int(h['units_2']), activation='tanh', recurrent_activation='sigmoid'))
        m.add(tf.keras.layers.Dropout(.2))
        m.add(tf.keras.layers.Dense(32, activation='relu'))
        m.add(tf.keras.layers.Dense(1, activation='linear'))
    else:
        raise ValueError(kind)
    return m


## 3. Tính số lượng tham số và kiểm tra công thức

Với input dimension `d`, RNN `u` hidden units: `u(d+u+1)`; LSTM: `4u(d+u+1)`; GRU Keras mặc định `reset_after=True`: `3u(d+u+2)`. Với `Dense(n)`: `(d+1)n`; Conv1D `(kernel*d+1)*filters`. Các công thức dùng để kiểm tra độc lập giá trị `count_params()`.

In [ ]:
def count_weights(model):
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    nontrainable = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return int(model.count_params()), trainable, nontrainable

def layer_detail(model, protocol, model_key, feature_set):
    return [dict(protocol=protocol, model=model_key, feature_set=feature_set,
                 layer_index=i, layer_type=layer.__class__.__name__,
                 layer_name=layer.name, params=int(layer.count_params()),
                 output_shape=str(getattr(layer,'output',None).shape) if hasattr(layer,'output') else '')
            for i,layer in enumerate(model.layers)]

def analytical_recurrent(kind, d, u):
    return {'LSTM': 4*u*(d+u+1), 'GRU':3*u*(d+u+2), 'SimpleRNN':u*(d+u+1)}[kind]

def check_architecture(model):
    # Independent analytical parameter formula with input shapes from weight tensors.
    expected = 0
    for layer in model.layers:
        t = type(layer).__name__
        if t in ('LSTM','GRU','SimpleRNN'):
            d = int(layer.kernel.shape[0])
            expected += analytical_recurrent(t,d,int(layer.units))
        elif t == 'Dense':
            expected += (int(layer.kernel.shape[0])+1)*int(layer.units)
        elif t == 'Conv1D':
            expected += (int(layer.kernel.shape[0])*int(layer.kernel.shape[1])+1)*int(layer.filters)
        elif t in ('InputLayer','Dropout','MaxPooling1D'):
            pass
        else:
            raise ValueError('Unexpected layer type: '+t)
    assert expected==model.count_params(), (model.name,expected,model.count_params())
    return expected

rows=[]; details=[]; models_for_summary={}
for feature_set, features in FEATURE_SETS_V20.items():
    d=len(features)
    for arch in ('lstm','gru','cnn_lstm','cnn_gru'):
        m=build_v20_net(arch,(LOOKBACK,d))
        check_architecture(m)
        total,trainable,nontrainable=count_weights(m)
        for objective in ('direct','residual'):
            key=f'{arch}_{objective}'
            rows.append(dict(protocol='V20',model=key,feature_set=feature_set,input_features=d,
                             input_shape=f'({LOOKBACK},{d})',total_params=total,trainable_params=trainable,
                             nontrainable_params=nontrainable,parameter_kind='neural network weights',notes='Same architecture for direct/residual'))
            details += layer_detail(m,'V20',key,feature_set)
        models_for_summary[('V20',feature_set,arch)]=m
    # Ridge(alpha=1), intercept fitted by sklearn: 22*d coefficients + 1 intercept.
    rows.append(dict(protocol='V20',model='ridge_residual',feature_set=feature_set,input_features=d,
                     input_shape=f'({LOOKBACK},{d}) flattened',total_params=LOOKBACK*d+1,
                     trainable_params=LOOKBACK*d+1,nontrainable_params=0,
                     parameter_kind='estimated regression coefficients',
                     notes='Ridge coef_.size + intercept_; not Keras count_params'))
    rows.append(dict(protocol='V20',model='naive_persistence',feature_set=feature_set,input_features=d,
                     input_shape='Copper(t-1)',total_params=0,trainable_params=0,nontrainable_params=0,
                     parameter_kind='no learned parameters',notes='Zero fitted parameters'))

for feature_set, features in FEATURE_SETS_V21.items():
    d=len(features); group=feature_set.split('_')[0]
    for kind in ('baseline_lstm','baseline_rnn','paper_sa_lstm','reopt_sa_lstm'):
        hp=REOPT_HP[group] if kind=='reopt_sa_lstm' else None
        m=build_v21_net(kind,(LOOKBACK,d),hp)
        check_architecture(m)
        total,trainable,nontrainable=count_weights(m)
        rows.append(dict(protocol='V21',model=kind,feature_set=feature_set,input_features=d,
                         input_shape=f'({LOOKBACK},{d})',total_params=total,trainable_params=trainable,
                         nontrainable_params=nontrainable,parameter_kind='neural network weights',
                         notes=f'SA units {hp["units_1"]}/{hp["units_2"]}' if hp else ('Paper SA 39/111; Dense(32) assumed' if kind=='paper_sa_lstm' else '')))
        details += layer_detail(m,'V21',kind,feature_set)
        models_for_summary[('V21',feature_set,kind)]=m
    rows.append(dict(protocol='V21',model='naive',feature_set=feature_set,input_features=d,
                     input_shape='Copper(t-1)',total_params=0,trainable_params=0,nontrainable_params=0,
                     parameter_kind='no learned parameters',notes='Zero fitted parameters'))

df=pd.DataFrame(rows)
layer_df=pd.DataFrame(details)
assert (df.total_params==df.trainable_params+df.nontrainable_params).all()
assert len(df)==3*(8+2)+4*(4+1)==50
# Verify V20 LSTM matches V21 baseline LSTM when input dimension matches
for fs20,fs21 in [('copper_paper3','paper3_with_copper'),('copper_legacy14','legacy14_with_copper')]:
    a=df.query('protocol=="V20" and feature_set==@fs20 and model=="lstm_direct"').total_params.iloc[0]
    b=df.query('protocol=="V21" and feature_set==@fs21 and model=="baseline_lstm"').total_params.iloc[0]
    assert a==b

csv_path=OUT/'model_parameter_counts_v20_v21.csv'
layer_path=OUT/'model_layer_parameter_details_v20_v21.csv'
df.to_csv(csv_path,index=False)
layer_df.to_csv(layer_path,index=False)
print('PASS:',len(df),'architecture/feature-set rows; all analytic parameter counts agree with Keras.')
print('Saved:',csv_path,layer_path)
display(df[['protocol','model','feature_set','input_features','total_params','trainable_params','nontrainable_params']])

## 4. Đọc từng tầng, hình dung độ phức tạp và xuất ZIP

`model.summary()` hiển thị số tham số của từng layer; chạy ở đây không `fit()` nên không có metric đánh giá. Mỗi mô hình đã học trong V20/V21 có cùng số tham số giữa các tỷ lệ train/test và ba chiến lược, nếu input và kiến trúc không đổi.

**Giới hạn quan trọng:** Bảng so sánh năng lực biểu diễn, không chứng minh model nhiều tham số là tốt hơn. `Ridge` đếm coefficients+intercept như các tham số ước lượng (khác Keras). Optimizer Adam có moment estimates nhưng không phải weights của model và không được tính.

In [ ]:
# Ví dụ chi tiết (có thể thay lựa chọn này)
chosen_protocol='V21'
chosen_feature_set='legacy14_with_copper'
chosen_model='reopt_sa_lstm'
chosen=models_for_summary[(chosen_protocol,chosen_feature_set,chosen_model)]
chosen.summary()
display(layer_df.query('protocol==@chosen_protocol and feature_set==@chosen_feature_set and model==@chosen_model'))

# Tạo biểu đồ tổng quan; Naive/Ridge vẫn có trong CSV
import matplotlib.pyplot as plt
fig, axs=plt.subplots(1,2,figsize=(16,6))
for ax,proto in zip(axs,['V20','V21']):
    sub=df[(df.protocol==proto) & (df.parameter_kind=='neural network weights')]
    for name,g in sub.groupby('model',sort=True):
        ax.scatter(g.input_features,g.total_params,label=name,alpha=.75)
    ax.set_title(f'{proto}: weights by input dimension')
    ax.set_xlabel('Number of input features')
    ax.set_ylabel('Trainable/total model parameters')
    ax.legend(fontsize=7,ncol=2)
    ax.grid(alpha=.2)
plt.tight_layout()
fig.savefig(OUT/'model_parameter_comparison.png',dpi=180,bbox_inches='tight')
plt.show()

# ZIP riêng, KHÔNG ghi đè các ZIP kết quả huấn luyện
from zipfile import ZipFile, ZIP_DEFLATED
archive=OUT.parent/'copper_v20_v21_model_parameters.zip'
with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('Download ZIP:',archive)
